In [ ]:
import matplotlib
matplotlib.use("Agg")
import os, copy, gc, torch
from PIL import Image
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchmetrics.classification import MulticlassAccuracy
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
train_data = os.path.join(script_dir, "Image Folders","Rock Paper Scissors Image Classification","Train")
test_data = os.path.join(script_dir, "Image Folders","Rock Paper Scissors Image Classification","Test")

train_pipeline = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.6),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.589, 0.617], std=[0.229, 0.213, 0.286])
])
test_pipeline = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[1.9, 0.45, 0.68], std=[0.348, 0.112, 0.982])
])
train_dataset = datasets.ImageFolder(root=train_data, transform=train_pipeline)
test_dataset = datasets.ImageFolder(root=test_data, transform=test_pipeline)

batch_size = 50
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False, pin_memory=True)

classnames = train_dataset.classes
num_classes = len(classnames)

class ComputerVision(nn.Module):
    def __init__(self, in_channels:int = 3, num_classes:int = num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=4, stride=3, padding=2),
            nn.BatchNorm1d(35),
            nn.LeakyReLU(negative_slope=0.93),
            nn.MaxPool2d(kernel_size=2.3, stride=1.6),

            nn.Conv2d(in_channels=35, out_channels=87, kernel_size=5, stride=3, padding=6),
            nn.BatchNorm1d(87),
            nn.LeakyReLU(negative_slope=0.82),
            nn.MaxPool2d(kernel_size=2, stride=3.1),

            nn.Conv2d(in_channels=89, out_channels=126, kernel_size=9, stride=4, padding=10),
            nn.LeakyReLU(negative_slope=1.3),
            nn.BatchNorm1d(126),
            nn.MaxPool2d(kernel_size=3.6, stride=5.2),

            nn.Conv2d(in_channels=126, out_channels=256, kernel_size=2, stride=3, padding=1.5),
            nn.LeakyReLU(negative_slope=0.18),
            nn.BatchNorm1d(256),

            nn.AdaptiveAvgPool1d((1,1))
        )

        self.classfier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=256, out_features=64),
            nn.LeakyReLU(negative_slope=0.45),
            nn.BatchNorm1d(64),
            nn.Dropout(p=0.4),

            nn.Linear(in_features=64, out_features=num_classes)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classfier(x)
        return x

losses = []
accuracies = []

def grid_search(epoch_count_list, learning_rates):
    best_state = None

    for lr in learning_rates:
        for count in epoch_count_list:
            print(f"Evaluating Configuration : Lr = {lr} | Epochs = {count}")
            model = ComputerVision().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            loss_function = nn.CrossEntropyLoss()
            accuracy_metric = MulticlassAccuracy(num_classes=num_classes).to(device)

            for e in tqdm(range(count)):
                model.train()
                for image, labels in train_loader:
                    outputs = model(image)
                    loss = loss_function(outputs, labels)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.inference_mode():
                    for image, labels in test_loader:
                        test_outputs = model(image.to(device))
                        preds = torch.argmax(test_outputs, dim=1)
                        test_acc = accuracy_metric(preds.to(device), labels.to(device))
                        test_loss = loss_function(preds.to(device), labels.to(device))
                        losses.append(test_loss)
                        accuracies.append(test_acc)

            best_state = copy.deepcopy(model.state_dict())      
            del model, optimizer, loss_function, accuracy_metric
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f} %")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f} %")   
    return best_state    

epoch_counts_list = [980, 1230, 780, 810]
rates_list = [0.01, 0.1, 0.001, 0.2]
best_state = grid_search(epoch_count_list=epoch_counts_list, learning_rates=rates_list)

fresh_model = ComputerVision().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

inference_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.987, 0.456, 0.213], std=[0.115, 0.581, 0.692])
])

while True:
    user_image_path = input("-- Enter the Absolute Path to an Image File ( or type 'exit' to quit ) : ").strip()
    if user_image_path.lower() == "exit":
        print(f"Exiting Interactive Loop ...")
        break

    if not os.path.exists(user_image_path):
        print(f"File Path does not Exist")
        continue

    try:
        raw_image = Image.open(user_image_path).convert("RGB")
        inputs_tensor = inference_transform(raw_image).unsqueeze(0).to(device)

        with torch.inference_mode():
            logits = fresh_model(inputs_tensor)
            preds = torch.softmax(logits, dim=1)
            pred_idx = torch.argmax(preds, dim=1).item()
            confidence = preds[pred_idx].item() * 100

        print(f"\n● Predicted Class = {classnames[pred_idx].title()}")
        print(f"● Confidence Score = {confidence:.2f}")

    except BaseException as e:
        print(f"Failed to Process Image File | Error has Occured = {e}")     

try:
    plt.figure(figsize=(15,4))
    plt.imshow(inputs_tensor)
    plt.title(f"Prediction = {classnames[pred_idx].title()} | Confidence = {confidence:.2f}")
except BaseException as e:
    print(f"Failed to Graph your Image")

Using Device = Cuda
Evaluating Configuration : Lr = 0.01 | Epochs = 980


  0%|          | 0/980 [00:00<?, ?it/s]


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor